# 05 — Hyperparameter Tuning

**Goal**: Improve model performance by finding better hyperparameters.

## Two strategies covered:

| Strategy | Tool | Best for |
|---|---|---|
| **Auto (evolutionary search)** | `model.tune()` | Full search, hands-off |
| **Manual targeted search** | Loop over key params | Quick wins on specific params |

## When to run this notebook:
- Evaluation Dice Score or mAP is below target
- Training curves show underfitting or overfitting
- You want to squeeze the last few percent of performance

## Priority order of hyperparameters to tune:
1. `lr0` + `lrf` — largest single impact on convergence
2. `batch` — affects generalization and gradient quality
3. `mosaic` + `copy_paste` — segmentation-specific augmentation strength
4. `weight_decay` — regularisation for small datasets
5. `imgsz` — larger = better mask quality but slower

> **Prerequisite**: Run `02_preprocessing.ipynb` first — `dataset/` must exist.

## 1. Imports & Setup

In [2]:
import torch
import yaml
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
import warnings
warnings.filterwarnings("ignore")

# .resolve() converts to absolute path — REQUIRED for tune() because it spawns
# separate Python subprocesses that may have a different working directory.
# Relative paths like "../dataset/dataset.yaml" will silently fail in subprocesses.
BASE_DIR     = Path("..").resolve()
DATASET_YAML = BASE_DIR / "dataset" / "dataset.yaml"

assert DATASET_YAML.exists(), "Run 02_preprocessing.ipynb first"

with open(DATASET_YAML) as f:
    ds = yaml.safe_load(f)

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print(f"GPU      : {'Yes — ' + torch.cuda.get_device_name(0) if DEVICE == 0 else 'No (CPU)'}")
print(f"Classes  : {ds['names']}")
print(f"BASE_DIR : {BASE_DIR}")       # must be an absolute path
print(f"Dataset  : {DATASET_YAML}")  # must be an absolute path

GPU      : Yes — NVIDIA GeForce RTX 4060 Laptop GPU
Classes  : ['class_1', 'class_2', 'class_3', 'class_4', 'class_5', 'class_6', 'class_7', 'class_8', 'class_9', 'class_10', 'class_11']
BASE_DIR : C:\NaufalFirdaus\CODES\MyGithub\self-driving-car-yolov26
Dataset  : C:\NaufalFirdaus\CODES\MyGithub\self-driving-car-yolov26\dataset\dataset.yaml


## Strategy A — Ultralytics Auto-Tune (Evolutionary Search)

Ultralytics uses a genetic/evolutionary algorithm to search over ~25 hyperparameters automatically.

**Recommended settings for our dataset:**
- `iterations=50` — sufficient for small search spaces; increase to 100 for more thorough search
- `epochs=30` — short per-trial to keep total time reasonable
- `use_ray=False` — simpler setup; set `True` if you have Ray Tune installed

**Expected runtime**: ~30 epochs × 50 trials = ~1500 epoch-equivalents. On an RTX 3060 this takes ~3–5 hours.

> Tip: Run this overnight. The best config is saved automatically.

In [3]:
# ── Auto-Tune Configuration ───────────────────────────────────────────────
TUNE_EPOCHS     = 20     # epochs per trial (short = fast screening)
TUNE_ITERATIONS = 30     # number of search trials
TUNE_PROJECT    = str(BASE_DIR / "runs" / "tune")
TUNE_NAME       = "yolo26s_tune1"

print(f"Auto-tune: {TUNE_ITERATIONS} trials × {TUNE_EPOCHS} epochs each")
print(f"Output   : {TUNE_PROJECT}/{TUNE_NAME}")
print("Starting evolutionary search...")

model = YOLO("yolo26s-seg.pt")

tune_results = model.tune(
    data        = str(DATASET_YAML),
    epochs      = TUNE_EPOCHS,
    iterations  = TUNE_ITERATIONS,
    imgsz       = 640,
    device      = DEVICE,
    optimizer   = "AdamW",
    amp         = True,
    use_ray     = False,     # set True if Ray Tune is installed
    project     = TUNE_PROJECT,
    name        = TUNE_NAME,
)

print("\n✓ Auto-tune complete.")

Auto-tune: 30 trials × 20 epochs each
Output   : C:\NaufalFirdaus\CODES\MyGithub\self-driving-car-yolov26\runs\tune/yolo26s_tune1
Starting evolutionary search...
Tuner: Initialized Tuner instance with 'tune_dir=C:\NaufalFirdaus\CODES\MyGithub\self-driving-car-yolov26\runs\tune\yolo26s_tune1-2'
Tuner:  Learn about tuning at https://docs.ultralytics.com/guides/hyperparameter-tuning
Tuner: Starting iteration 1/30 with hyperparameters: {'lr0': 0.01, 'lrf': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005, 'warmup_epochs': 3.0, 'warmup_momentum': 0.8, 'box': 7.5, 'cls': 0.5, 'cls_pw': 0.0, 'dfl': 1.5, 'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4, 'degrees': 0.0, 'translate': 0.1, 'scale': 0.5, 'shear': 0.0, 'perspective': 0.0, 'flipud': 0.0, 'fliplr': 0.5, 'bgr': 0.0, 'mosaic': 1.0, 'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0, 'close_mosaic': 10}
Saved C:\NaufalFirdaus\CODES\MyGithub\self-driving-car-yolov26\runs\tune\yolo26s_tune1-2\tune_scatter_plots.png
Saved C:\NaufalFirdaus\CODES\MyGit

## Read Auto-Tune Results

In [4]:
tune_dir = Path(TUNE_PROJECT) / TUNE_NAME
best_yaml = tune_dir / "best_hyperparameters.yaml"

if best_yaml.exists():
    with open(best_yaml) as f:
        best_params = yaml.safe_load(f)
    print("=== Best Hyperparameters Found ===")
    for k, v in best_params.items():
        print(f"  {k:<25} {v}")
else:
    print(f"best_hyperparameters.yaml not found at {best_yaml}")
    print("Check the tune output directory manually.")
    best_params = {}

# Plot tune_results.csv if available
tune_csv = tune_dir / "tune_results.csv"
if tune_csv.exists():
    df = pd.read_csv(tune_csv)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(df.index, df["fitness"], marker="o", markersize=3, linewidth=1, color="steelblue")
    axes[0].set_xlabel("Trial")
    axes[0].set_ylabel("Fitness (higher = better)")
    axes[0].set_title("Fitness per Trial")
    axes[0].axhline(df["fitness"].max(), linestyle="--", color="red", label=f"Best: {df['fitness'].max():.4f}")
    axes[0].legend()

    axes[1].plot(df.index, df["fitness"].cummax(), linewidth=2, color="green")
    axes[1].set_xlabel("Trial")
    axes[1].set_ylabel("Best Fitness So Far")
    axes[1].set_title("Best Fitness Cumulative")

    plt.tight_layout()
    plt.savefig("tune_fitness.png", dpi=150)
    plt.show()
    print("Saved: tune_fitness.png")

=== Best Hyperparameters Found ===
  lr0                       0.01
  lrf                       0.01
  momentum                  0.937
  weight_decay              0.0005
  warmup_epochs             3.0
  warmup_momentum           0.8
  box                       7.5
  cls                       0.5
  cls_pw                    0.0
  dfl                       1.5
  hsv_h                     0.015
  hsv_s                     0.7
  hsv_v                     0.4
  degrees                   0.0
  translate                 0.1
  scale                     0.5
  shear                     0.0
  perspective               0.0
  flipud                    0.0
  fliplr                    0.5
  bgr                       0.0
  mosaic                    1.0
  mixup                     0.0
  cutmix                    0.0
  copy_paste                0.0
  close_mosaic              10


## Strategy B — Manual Targeted Grid Search

Use this when you want to understand the **sensitivity** of specific parameters
or when auto-tune is too slow.

We sweep the three highest-impact parameters: `lr0`, `weight_decay`, and `mosaic`.

In [5]:
# ── Manual Grid Configuration ─────────────────────────────────────────────
# Each dict defines one experiment. Keep SHORT_EPOCHS low for screening.
SHORT_EPOCHS = 30

SEARCH_GRID = [
    # (lr0,   weight_decay, mosaic,  run_name)
    (1e-3,  5e-4,  0.8,  "grid_lr1e3_wd5e4_mo0.8"),   # baseline
    (5e-4,  5e-4,  0.8,  "grid_lr5e4_wd5e4_mo0.8"),   # lower lr
    (1e-3,  1e-3,  0.8,  "grid_lr1e3_wd1e3_mo0.8"),   # higher weight_decay
    (1e-3,  5e-4,  1.0,  "grid_lr1e3_wd5e4_mo1.0"),   # full mosaic
    (1e-3,  5e-4,  0.5,  "grid_lr1e3_wd5e4_mo0.5"),   # lighter mosaic
]

print(f"Grid experiments : {len(SEARCH_GRID)}")
print(f"Epochs per run   : {SHORT_EPOCHS}")
print(f"Total epochs     : {len(SEARCH_GRID) * SHORT_EPOCHS}")

Grid experiments : 5
Epochs per run   : 30
Total epochs     : 150


In [6]:
GRID_PROJECT = str(BASE_DIR / "runs" / "grid_search")
grid_results = []

for lr0, wd, mosaic, run_name in SEARCH_GRID:
    print(f"\nRunning: {run_name}  (lr0={lr0}, wd={wd}, mosaic={mosaic})")
    m = YOLO("yolo26s-seg.pt")
    r = m.train(
        data         = str(DATASET_YAML),
        epochs       = SHORT_EPOCHS,
        imgsz        = 640,
        device       = DEVICE,
        lr0          = lr0,
        weight_decay = wd,
        mosaic       = mosaic,
        optimizer    = "AdamW",
        amp          = True,
        cos_lr       = True,
        patience     = 10,
        project      = GRID_PROJECT,
        name         = run_name,
        verbose      = False,
    )
    seg_map50 = r.results_dict.get("metrics/mAP50(M)", 0.0)
    grid_results.append({
        "run_name"    : run_name,
        "lr0"         : lr0,
        "weight_decay": wd,
        "mosaic"      : mosaic,
        "seg_mAP50"   : round(float(seg_map50), 4),
    })
    print(f"  → Seg mAP@50 = {seg_map50:.4f}")

print("\n✓ Grid search complete.")


Running: grid_lr1e3_wd5e4_mo0.8  (lr0=0.001, wd=0.0005, mosaic=0.8)
Ultralytics 8.4.41  Python-3.12.10 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\NaufalFirdaus\CODES\MyGithub\self-driving-car-yolov26\dataset\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s-seg.pt, momentum=0.

2026/04/25 13:20:55 INFO mlflow.tracking.fluent: Experiment with name 'C:\NaufalFirdaus\CODES\MyGithub\self-driving-car-yolov26\runs\grid_search' does not exist. Creating a new experiment.


MLflow: logging run_id(ed9da6030ec049c080865b654214c02f) to runs\mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs\mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to C:\NaufalFirdaus\CODES\MyGithub\self-driving-car-yolov26\runs\grid_search\grid_lr1e3_wd5e4_mo0.8
Starting training for 30 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       1/30       8.5G      2.033      3.724      3.881    0.01296      8.928        312        640: 100% ━━━━━━━━━━━━ 19/19 3.4s/it 1:052.8sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.2s/it 6.7s1.5s1s
                   all         74       1956    0.00152     0.0251    0.00041   8.38e-05      0.185     0.0013   0.000177   2.66e-05

      Epoch    GPU_mem  

In [7]:
# Summarize grid results
df_grid = pd.DataFrame(grid_results).sort_values("seg_mAP50", ascending=False)
print("=== Grid Search Results (sorted by Seg mAP@50) ===")
print(df_grid.to_string(index=False))

best_row = df_grid.iloc[0]
print(f"\n✓ Best config: lr0={best_row.lr0}, weight_decay={best_row.weight_decay}, mosaic={best_row.mosaic}")
print(f"  Seg mAP@50 = {best_row.seg_mAP50}")

# Save to JSON
df_grid.to_json(BASE_DIR / "grid_search_results.json", orient="records", indent=2)

=== Grid Search Results (sorted by Seg mAP@50) ===
              run_name    lr0  weight_decay  mosaic  seg_mAP50
grid_lr1e3_wd5e4_mo0.8 0.0010        0.0005     0.8     0.6457
grid_lr1e3_wd1e3_mo0.8 0.0010        0.0010     0.8     0.6419
grid_lr1e3_wd5e4_mo0.5 0.0010        0.0005     0.5     0.6328
grid_lr5e4_wd5e4_mo0.8 0.0005        0.0005     0.8     0.6244
grid_lr1e3_wd5e4_mo1.0 0.0010        0.0005     1.0     0.6228

✓ Best config: lr0=0.001, weight_decay=0.0005, mosaic=0.8
  Seg mAP@50 = 0.6457


In [8]:
# Plot grid results
fig, ax = plt.subplots(figsize=(10, 5))
colors  = ["#2ecc71" if v == df_grid["seg_mAP50"].max() else "#3498db" for v in df_grid["seg_mAP50"]]
ax.barh(df_grid["run_name"], df_grid["seg_mAP50"], color=colors, edgecolor="#333")
ax.set_xlabel("Seg mAP@50", fontsize=12)
ax.set_title("Grid Search: Seg mAP@50 per Configuration", fontsize=13)
for i, (_, row) in enumerate(df_grid.iterrows()):
    ax.text(row["seg_mAP50"] + 0.002, i, f"{row['seg_mAP50']:.4f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("grid_search_results.png", dpi=150)
plt.show()
print("Saved: grid_search_results.png")

<Figure size 1000x500 with 1 Axes>

Saved: grid_search_results.png


## Final Training with Best Configuration
Apply the best hyperparameters found above and train for full epochs.

In [9]:
# ── Option A: use best_params from auto-tune ──────────────────────────────
# If you ran Strategy A, the best_hyperparameters.yaml is automatically used
# by YOLO when you pass it as the 'cfg' argument.
#
# ── Option B: use best params from grid search ─────────────────────────────
# Manually set the best values found in Strategy B.

# Read best from grid results (or override manually)
try:
    best_lr0 = float(best_row.lr0)
    best_wd  = float(best_row.weight_decay)
    best_mos = float(best_row.mosaic)
except NameError:
    # Fallback if grid search was skipped
    best_lr0, best_wd, best_mos = 1e-3, 5e-4, 0.8

print(f"Final training config:")
print(f"  lr0          : {best_lr0}")
print(f"  weight_decay : {best_wd}")
print(f"  mosaic       : {best_mos}")
print(f"  epochs       : 150  (full training)")

Final training config:
  lr0          : 0.001
  weight_decay : 0.0005
  mosaic       : 0.8
  epochs       : 150  (full training)


In [ ]:
final_model = YOLO("yolo26s-seg.pt")

final_results = final_model.train(
    data         = str(DATASET_YAML),
    epochs       = 150,
    imgsz        = 640,
    batch        = -1,
    device       = DEVICE,
    optimizer    = "AdamW",
    lr0          = best_lr0,
    lrf          = 0.01,
    weight_decay = best_wd,
    mosaic       = best_mos,
    mixup        = 0.1,
    copy_paste   = 0.1,
    amp          = True,
    cos_lr       = True,
    patience     = 30,
    seed         = 42,
    project      = str(BASE_DIR / "runs" / "segment"),
    name         = "yolo26s_final",
    save         = True,
    plots        = True,
    verbose      = True,
)

final_best = BASE_DIR / "runs" / "segment" / "yolo26s_final" / "weights" / "best.pt"
print(f"\n✓ Final model saved: {final_best}")
print(f"Seg mAP@50    : {final_results.results_dict.get('metrics/mAP50(M)', 'N/A'):.4f}")
print(f"Seg mAP@50-95 : {final_results.results_dict.get('metrics/mAP50-95(M)', 'N/A'):.4f}")
print("\n✓ Update WEIGHTS_PATH in 04_evaluation.ipynb to point to yolo26s_final/weights/best.pt")

Ultralytics 8.4.41  Python-3.12.10 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\NaufalFirdaus\CODES\MyGithub\self-driving-car-yolov26\dataset\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo26s-seg.pt, momentum=0.937, mosaic=0.8, multi_scale=0.0, name=yolo26s_final-2, nbs=64, nms=

## Hyperparameter Reference

| Parameter | Default | Range to try | Notes |
|---|---|---|---|
| `lr0` | 1e-3 | 5e-4 – 5e-3 | Reduce if loss oscillates |
| `lrf` | 0.01 | 0.005 – 0.1 | Final LR fraction |
| `weight_decay` | 5e-4 | 1e-4 – 1e-3 | Increase for overfitting |
| `warmup_epochs` | 3 | 3 – 5 | Increase for larger LR |
| `mosaic` | 0.8 | 0.5 – 1.0 | Key for small-object perf |
| `copy_paste` | 0.1 | 0.0 – 0.3 | Segmentation-specific |
| `mixup` | 0.1 | 0.0 – 0.2 | Mild regularisation |
| `degrees` | 5.0 | 0 – 15 | More rotation for varied scenes |
| `scale` | 0.5 | 0.3 – 0.7 | Scale jitter range |
| `imgsz` | 640 | 480, 640, 768 | Larger → better masks, slower |
| `batch` | auto | 8 – 32 | Larger = more stable gradients |